# Comparative epitope mapping

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoLlama/epitope_align/blob/claude/antibody-epitope-identification-ervnyf/notebooks/epitope_mapping.ipynb)

Work out **where an antibody probably binds** from the fact that it binds some
species' versions of a protein and not others.

You do not need to know how to code. You will edit a few lines of text between
quote marks, and press the **▶ play button** to the left of each grey box.

> ### First: click **Copy to Drive** at the top of this page
>
> A notebook opened straight from GitHub is a read-only preview, so your typing
> may not stick. **Copy to Drive** gives you your own editable copy, and
> anything you change is saved there. If you do not see the button, use
> **File → Save a copy in Drive**.

Then work down the page in order:

1. **Set up** - installs the tool (about two minutes, once per session)
2. **Check it works** - runs a built-in example with a known answer
3. **Enter your data** - edit the text between the quote marks
4. **Run and read the results**

Nothing here touches your own computer, and nothing you type is sent anywhere
except to UniProt and the structure databases, to look up the entries you name.

---

## Step 1 - Set up

Nothing to edit here. Press ▶ and wait for the word **Ready**. It installs the
tool from GitHub along with two optional helpers (MAFFT for better
multi-species alignments, DSSP for secondary structure).


In [ ]:
# Setup - nothing to edit. Press play and wait for 'Ready'.
# Safe to re-run: it always reinstalls the current code from GitHub.

REPO = 'https://github.com/NanoLlama/epitope_align.git'
BRANCH = 'claude/antibody-epitope-identification-ervnyf'  # change to 'main' once this branch is merged

import importlib
import subprocess
import sys

# three genuinely different aligners, not three settings of one: agreement
# between MAFFT settings is not evidence, so alignment_confidence stays blank
# until at least three independent programs are available
print('Installing MAFFT, MUSCLE, Clustal Omega and DSSP ...')
subprocess.run('apt-get -qq update', shell=True, capture_output=True)
subprocess.run(
    'apt-get -qq install -y mafft muscle clustalo dssp',
    shell=True, capture_output=True,
)

print('Installing epitope-map ...')
source = f'git+{REPO}@{BRANCH}'
for command in (
    # first pass: the package and its dependencies
    f'pip install -q "{source}"',
    # second pass: force the code itself to be replaced. Without this pip sees
    # the same version number already installed and keeps the old copy, so a
    # re-run silently gives you yesterday's build.
    f'pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir "{source}"',
):
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise SystemExit(
            'Install failed. Check the branch name above still exists on GitHub.'
        )

# drop anything imported from the previous build, or Python keeps serving it
for name in [n for n in sys.modules if n == 'epitope_map' or n.startswith('epitope_map.')]:
    del sys.modules[name]
importlib.invalidate_caches()

import epitope_map
from epitope_map import notebook as _notebook_helpers  # must exist in a current build

MIN_VERSION = (0, 4, 0)
installed = tuple(int(part) for part in epitope_map.__version__.split('.')[:3])
if installed < MIN_VERSION:
    raise SystemExit(
        f'epitope-map {epitope_map.__version__} is installed but this notebook '
        f'needs {".".join(str(p) for p in MIN_VERSION)} or newer. The runtime is '
        'holding an older copy: choose Runtime -> Restart session, then run this '
        'cell again.'
    )

def _have(tool):
    return 'yes' if subprocess.run(f'which {tool}', shell=True,
                                   capture_output=True).returncode == 0 else 'no'

print(f'\nReady: epitope-map {epitope_map.__version__}')
print('installed from:', epitope_map.__file__)
aligners = [t for t in ('mafft', 'muscle', 'clustalo') if _have(t) == 'yes']
print('aligners:', ', '.join(aligners) if aligners else 'none')
print('DSSP:    ', _have('mkdssp'))
if len(aligners) < 3:
    print(
        f'\nNOTE: {len(aligners)} of 3 aligners installed. alignment_confidence '
        'will be reported as blank rather than as a number, because agreement '
        'between fewer than three independent methods is not evidence.'
    )


---

## Step 2 - Check it works

This runs a made-up example where the answer is known in advance: six invented
species, three that bind and three that do not, and an epitope deliberately
planted at residues **65, 66, 68, 70, 72, 114, 116 and 118**.

If the top patch printed below is exactly those eight residues, everything is
working.


In [ ]:
!epitope-map --demo --outdir /content/demo-run


---

## Step 3 - Enter your data

In the box below, replace the text **between the quote marks** with your own,
leaving the quote marks in place. Then press ▶. Nothing is analysed yet - the
cell only checks that what you typed makes sense and prints it back, so you can
confirm it was read correctly before anything runs.

**`sequences`** - your species and their UniProt accession numbers, as
`label=ACCESSION`, separated by commas:

```
mouse=Q61503, rat=P21590, human=P21589
```

Look each one up at <https://www.uniprot.org> by searching your protein plus the
species name; the accession is the code like `P21589` near the top of the entry.
The labels are yours to choose - they only have to match the binding line.

**`binding`** - the same labels with their results, on one line:

```
mouse=binder, rat=binder, human=non_binder, marmoset=non_binder
```

Semicolons work as well as commas, `mouse,binder; rat,binder` works if you
prefer, and `yes`/`no` are accepted. Use `unknown` for a species you have a
sequence for but no binding data - it will be shown but not scored.

You need at least one binder and one non-binder. Three scored species is the
minimum; six to eight is far better, because each extra informative species
roughly halves the shortlist.

**`reference`** - the species the structure belongs to. It must be one that
binds: all numbering in the results refers to it.

**`structure`** - an AlphaFold accession like `AF-P21589-F1` (use the reference
species' UniProt accession; every UniProt page links its AlphaFold model), or a
Protein Data Bank ID like `4H2I` if an experimental structure exists. To use
your own file, set `upload_my_own_structure = True` and you will be asked for it.

**`topology`** - which part of the protein is outside the cell. An antibody can
only reach that part, so the tool refuses to run without knowing it rather than
ranking residues inside the cell (that mistake produced a confident-looking
rank-2 hit on the first real target). Leave it as `"auto"` and it is read from
the UniProt entry, which works whenever you gave accessions above. If UniProt
has no topology for your protein the run will stop and tell you; then either
write it out - `"extracellular=121-763,tm=68-88,cytoplasmic=1-67"` - or, for a
soluble protein or sequences you already trimmed to the ectodomain, put
`"whole-chain"`.

**`ectodomain`** - optional. If you only care about part of the protein - say
the part outside the cell - write it as `"25-240"`, in the structure's own
numbering. Leave as `""` to analyse everything.


In [ ]:
# =====================================================================
#  EDIT THE TEXT BETWEEN THE QUOTE MARKS BELOW, THEN PRESS PLAY.
#  Keep the quote marks themselves. Nothing is analysed by this cell -
#  it only checks what you typed and prints it back.
# =====================================================================

# Your species and their UniProt accessions, as label=ACCESSION
sequences = "mouse=REPLACE_ME, rat=REPLACE_ME, human=REPLACE_ME"

# Which of them the antibody binds
binding = "mouse=binder, rat=binder, human=non_binder"

# The species the structure belongs to. Must be one that binds.
reference = "mouse"

# AlphaFold accession (AF-<uniprot>-F1), or a PDB ID like 4H2I
structure = "AF-REPLACE_ME-F1"

# Which part of the protein sticks out of the cell. An antibody can only reach
# that part, so this is required. Leave as "auto" to read it from UniProt (works
# whenever you gave accessions above). For a soluble protein, or if you already
# trimmed your sequences to the ectodomain, use "whole-chain".
topology = "auto"

# A built-in target profile fills in sequences, topology, structural domains and
# candidate orthologs for a protein you analyse repeatedly. "tfr1" is installed.
# Anything you set explicitly above wins over the profile. Leave "" for none.
target = "tfr1"

# Structural domains, as a TSV of name/start/end. Leave "" to use the profile's,
# or to let the model's contact graph be decomposed when there is no profile.
domains = ""

# Candidate orthologs to price for the panel advice: a FASTA path, or
# "label=ACCESSION" pairs separated by commas. Leave "" to use the profile's.
candidate_species = ""

# Re-cluster at several radii to see which patches are really one surface.
radius_sweep = "10,12,14,16,18"

# A previous run's output directory, to map its patch IDs onto this run's.
compare_run = ""

# Optional. Leave as "" to analyse the whole chain / let it pick the chain.
ectodomain = ""          # e.g. "25-240", in the structure's own numbering
chain = ""               # e.g. "A"

# Set either to True to upload your own file instead of fetching one
upload_my_own_fasta = False
upload_my_own_structure = False

# =====================================================================
#  Nothing below this line needs editing.
# =====================================================================
import pathlib

try:
    from epitope_map.io_seq import InputError
    from epitope_map.notebook import (
        check_panel, parse_binding_calls, sequence_labels, summarise, write_binding_csv,
    )
except ImportError:
    raise SystemExit(
        'This is an older build of epitope-map than the notebook expects. '
        'Re-run Step 1 (the setup cell) and then this cell again.'
    )

work = pathlib.Path('/content/my-run')
work.mkdir(parents=True, exist_ok=True)

# a fresh numbered directory per run, so a previous run is never overwritten and
# the two can be diffed
existing = sorted(int(p.name.split('-')[1]) for p in work.glob('run-*') if p.name.split('-')[-1].isdigit())
run_number = (existing[-1] + 1) if existing else 1
results_dir = work / f'run-{run_number}'
print(f'results will be written to {results_dir}\n')

if upload_my_own_fasta:
    from google.colab import files
    print('Choose your FASTA file:')
    uploaded = files.upload()
    sequences_arg = str(work / list(uploaded)[0])
    pathlib.Path(sequences_arg).write_bytes(list(uploaded.values())[0])
else:
    sequences_arg = sequences

if upload_my_own_structure:
    from google.colab import files
    print('Choose your structure file (.pdb or .cif):')
    uploaded = files.upload()
    structure_arg = str(work / list(uploaded)[0])
    pathlib.Path(structure_arg).write_bytes(list(uploaded.values())[0])
else:
    structure_arg = structure

if target.strip() and not sequences.strip():
    sequences_arg = ''  # the profile supplies them

if 'REPLACE_ME' in sequences_arg or 'REPLACE_ME' in structure_arg:
    raise SystemExit(
        'The lines above still say REPLACE_ME. Put your own UniProt accessions '
        'in, then press play again.'
    )

try:
    calls = parse_binding_calls(binding)
except InputError as problem:
    raise SystemExit(f'Could not read the binding line: {problem}')

binding_path = write_binding_csv(calls, work / 'binding.csv')
problems, notes = check_panel(calls, sequence_labels(sequences_arg), reference)

print(summarise(calls, reference))
print(f'\nstructure : {structure_arg}')
print(f'sequences : {sequences_arg or f"from the {target} target profile"}')
print(f'range     : {ectodomain.strip() or "whole chain"}')

for note in notes:
    print(f'\nnote: {note}')
if problems:
    print('\n' + '=' * 60)
    for problem in problems:
        print(f'FIX THIS: {problem}')
    print('=' * 60)
    raise SystemExit('Correct the lines above and press play again.')
print('\nInputs look consistent. Move on to Step 4.')


---

## Step 4 - Run it

This fetches anything it needs, aligns the sequences, measures the surface of the
structure and ranks the candidate patches. A small protein takes under a minute.

Read the **warnings** it prints. They are not errors - they are the honest
limitations of your particular run, and the biggest one is usually that your
binders and non-binders are two separate branches of the family tree, which
leaves a lot of irrelevant differences looking meaningful.


In [ ]:
# Nothing to edit. Assembles the command, checks that every box above reached
# it, and runs.
import shlex, subprocess

command = ['epitope-map']

# (variable, flag, include-when-empty?) - one row per box in Step 3
wiring = [
    (sequences_arg, '--sequences'),
    (str(binding_path), '--binding'),
    (reference, '--reference'),
    (structure_arg, '--structure'),
    (target, '--target'),
    (topology if topology.strip().lower() != 'auto' else '', '--topology'),
    (ectodomain, '--ectodomain'),
    (chain, '--chain'),
    (domains, '--domains'),
    (candidate_species, '--candidate-species'),
    (radius_sweep, '--radius-sweep'),
    (compare_run, '--compare-run'),
]
for value, flag in wiring:
    if str(value).strip():
        command += [flag, str(value).strip()]
command += ['--outdir', str(results_dir)]

# every non-empty box must appear in the command: a variable that silently never
# reaches the CLI is how four runs went out without their candidate species
missing = [
    flag for value, flag in wiring
    if str(value).strip() and flag not in command
]
if missing:
    raise SystemExit(f'these settings did not reach the command line: {missing}')

print(' '.join(shlex.quote(part) for part in command), '\n')
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('--- it stopped with this message ---')
    print(result.stderr)


---

## Step 5 - Read the results

The report below is the thing to read, and it is ordered so the things that
would discredit a hit come before the hit. Section 1 says what was excluded and
why, how much signal there is at all, and where in the chain the divergence
sits; section 5 is the ranked answer; section 7 is what to make in the lab.

Two things worth checking in section 1 before you believe anything: whether the
binding pattern simply follows the family tree (if so the shortlist is weak),
and whether any region - a stalk, a linker - is far more "discriminating" than
the rest of the chain, which usually means it is unconstrained rather than
important.

Remember what this is: a **ranked shortlist of hypotheses to test**, not a
prediction. The top patch being right is something your chimeras and mutants
decide, not the software.


In [ ]:
from IPython.display import Markdown, display
import pandas as pd, pathlib

results = results_dir
display(Markdown((results / 'report.md').read_text()))


### The candidate patches, as a table


In [ ]:
patches = pd.read_csv(results / 'patches.tsv', sep='\t')
display(patches[['patch_id', 'rank_raw', 'rank_normalized', 'n_residues',
                 'residues', 'total_score', 'mean_rsa', 'spread_A', 'flags']])


### The mutants worth making

`gain_of_binding` rows are the convincing experiment: they put the binder's
residue into a species that does *not* bind, so a positive result cannot be
explained away as a badly folded protein.


In [ ]:
mutants = pd.read_csv(results / 'mutants.tsv', sep='\t')
display(mutants[['patch_id', 'direction', 'background_species', 'mutation',
                 'numbering', 'grantham', 'rsa', 'priority']].head(20))


### Download everything

Includes `session.pml`, which opens the structure in PyMOL with the candidate
patches coloured in.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(f'/content/epitope-results-run-{run_number}', 'zip', results)
files.download(f'/content/epitope-results-run-{run_number}.zip')
